#### Relevant Imports

In [1]:
from sqlalchemy import create_engine, text
import json
from collections import defaultdict
from typing import List, Dict, Any
import csv
from groq import Groq
import dotenv
from google import genai
from google.genai import types
import os

**SQLite DB**  
Let us connect to the SQLite Sample Database what we have  


In [2]:
# There is an engine instance created, which can handle multiple connetions
DB_File = "Sample_1 - Copy.db"

if os.path.exists (DB_File):
    sql_engine = create_engine("sqlite:///"+DB_File)
    conn_1 = sql_engine.connect ()
else:
    print ("DB Files does not exist")

**Output in JSON**  
From a specific search Criteria, SQL query is executed to get the results  
Further this is used as context to LLM to answer a question

In [3]:
# Get the details of Service Rep for customers in a country 
Country = 'USA'
result = conn_1.execute (text(f"""
                                SELECT
                                    c.FirstName || ' ' || c.LastName AS CustomerName,
                                    c.State AS CustomerState,
                                    c.Country AS CustomerCountry,
                                    e.FirstName || ' ' || e.LastName AS SupportRepName,
                                    e.State AS RepState,
                                    e.Country AS RepCountry,
                                    e.Email AS SupportRepEmail
                                FROM Customer c
                                LEFT JOIN Employee e ON c.SupportRepId = e.EmployeeId
                                WHERE c.Country = '{Country}';  
                              """))

# Query output into JSON
rows = [row._asdict () for row in result]

# Convert to JSON
json_output = json.dumps(rows,  indent=2)
print (json_output)


[
  {
    "CustomerName": "Frank Harris",
    "CustomerState": "CA",
    "CustomerCountry": "USA",
    "SupportRepName": "Margaret Park",
    "RepState": "AB",
    "RepCountry": "Canada",
    "SupportRepEmail": "margaret@chinookcorp.com"
  },
  {
    "CustomerName": "Jack Smith",
    "CustomerState": "WA",
    "CustomerCountry": "USA",
    "SupportRepName": "Steve Johnson",
    "RepState": "AB",
    "RepCountry": "Canada",
    "SupportRepEmail": "steve@chinookcorp.com"
  },
  {
    "CustomerName": "Michelle Brooks",
    "CustomerState": "NY",
    "CustomerCountry": "USA",
    "SupportRepName": "Jane Peacock",
    "RepState": "AB",
    "RepCountry": "Canada",
    "SupportRepEmail": "jane@chinookcorp.com"
  },
  {
    "CustomerName": "Tim Goyer",
    "CustomerState": "CA",
    "CustomerCountry": "USA",
    "SupportRepName": "Jane Peacock",
    "RepState": "AB",
    "RepCountry": "Canada",
    "SupportRepEmail": "jane@chinookcorp.com"
  },
  {
    "CustomerName": "Dan Miller",
    "Custom

**Use it in LLM**
Define a Groq Client and invoke LLM to answer question  
use the fetched information as context for query

In [5]:
# Load Env variable and define a Groq Client
dotenv.load_dotenv ()
client_gr = Groq()
# client_gr = Groq (api_key='Your Key')
model_gr = "llama-3.3-70b-versatile"

client_gm = genai.Client()
# client_gm = genai.Client(api_key="Your Key")
model_gm = "gemini-2.5-flash-lite"

> Invoke of Groq API for llama model  
> Specific instructions are required

In [6]:
Instr = "You are given a user query and a context in JSON format.\
        Using the context given, provide response to the user question or statement.\
        Answer to the question precisely"

Question = f"How many services reps the {Country} customers have?"
# Question = f"Who can the {Country} customers contact if they have an issue?"
# Question = f"Are the service reps in same place of the customers?"
# Question = f"Tell me how much time diff b/w victor and his service guy"
# Question = f"Who handles more customers?"

messages=[
    {
        "role": "system",
        "content": Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ json_output + "Query : \n" + Question
    }
]
completion = client_gr.chat.completions.create(
    messages=messages,
    # model=model_gr,
    model='openai/gpt-oss-120b'
)

print (completion.choices[0].message.content)

The USA customers are served by **three** different support representatives: Margaret Park, Steve Johnson, and Jane Peacock.


> With the same context details, check the response from Gemini LLM

In [9]:
# Question = f"Who can the {Country} customers contact if they have an issue?"
# Question = f"Are the service reps in same place of the customers?"
Question = f"Tell me how much time diff b/w victor and his service guy"

# Invoke Gemini LLM API
response = client_gm.models.generate_content(
                model=model_gm,
                config =types.GenerateContentConfig(
                            system_instruction=Instr,
                            ),
                contents = "Context : \n"+ json_output + "\nUser Query : \n"+Question
)

print(response.text)

I cannot provide the time difference between Victor Stevens and his service representative. The provided data does not contain information about time zones or any time-related details.


> Additional details for manager is also fetched  
> Only contact details are gathered 
> The retrived data is processed as text to provide in context

In [10]:
# Get the details of Service Rep for customers in a country including the manager details
Country = 'Canada'
result = conn_1.execute (text(f"""
                                SELECT
                                    c.FirstName || ' ' || c.LastName AS CustomerName,
                                    c.State AS CustomerState,
                                    e.FirstName || ' ' || e.LastName AS SupportRepName,
                                    e.Email AS SupportRepEmail,
                                    m.FirstName || ' ' || m.LastName AS ManagerName,
                                    m.Email AS ManagerEmail
                                FROM Customer c
                                LEFT JOIN Employee e ON c.SupportRepId = e.EmployeeId
                                LEFT JOIN Employee m ON e.ReportsTo = m.EmployeeId
                                WHERE c.Country = '{Country}';
                              """))

# Make each row into a sentence
rows = result.fetchall()
context = []
for row in rows:
    Customer, C_State, Rep, Rep_Mail, Manager, Mgr_Mail = row
    sentence = f"{Customer} from state {C_State} may contact sales rep {Rep} @ {Rep_Mail}, who reports to the manager {Manager} @ {Mgr_Mail}"
    context.append(sentence)

for line in context: 
  print (line)

François Tremblay from state QC may contact sales rep Jane Peacock @ jane@chinookcorp.com, who reports to the manager Nancy Edwards @ nancy@chinookcorp.com
Mark Philips from state AB may contact sales rep Steve Johnson @ steve@chinookcorp.com, who reports to the manager Nancy Edwards @ nancy@chinookcorp.com
Jennifer Peterson from state BC may contact sales rep Jane Peacock @ jane@chinookcorp.com, who reports to the manager Nancy Edwards @ nancy@chinookcorp.com
Robert Brown from state ON may contact sales rep Jane Peacock @ jane@chinookcorp.com, who reports to the manager Nancy Edwards @ nancy@chinookcorp.com
Edward Francis from state ON may contact sales rep Jane Peacock @ jane@chinookcorp.com, who reports to the manager Nancy Edwards @ nancy@chinookcorp.com
Martha Silk from state NS may contact sales rep Steve Johnson @ steve@chinookcorp.com, who reports to the manager Nancy Edwards @ nancy@chinookcorp.com
Aaron Mitchell from state MB may contact sales rep Margaret Park @ margaret@chi

In [11]:
Instr = "You are given a user query and context details.\
        Using the context given, provide response to the user question or statement.\
        Answer to the question precisely"

Question = f"Tell me how is the escalation matrix organised"
# Question = f"Who can the {Country} customers contact if the service support is not satisfactory"
# Question = f"How many Reps report to the managers?"

messages=[
    {
        "role": "system",
        "content": Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ "".join(context) + "\nQuery : \n" + Question
    }
]
completion = client_gr.chat.completions.create(
    messages=messages,
    model=model_gr,
)

print ("Groq Response : ",completion.choices[0].message.content)

# Invoke Gemini LLM API
response = client_gm.models.generate_content(
                model=model_gm,
                # model="gemini-2.5-flash",
                config =types.GenerateContentConfig(
                            system_instruction=Instr,
                            ),
                contents = "Context : \n"+ "".join(context) + "\nUser Query : \n"+Question
)

print("Gemini Response :",response.text)

Groq Response :  The escalation matrix is organized as follows:

1. **Sales Representatives**: 
   - Jane Peacock (jane@chinookcorp.com) handles QC, BC, ON (Robert Brown and Edward Francis), and NT (Ellie Sullivan).
   - Steve Johnson (steve@chinookcorp.com) handles AB (Mark Philips) and NS (Martha Silk).
   - Margaret Park (margaret@chinookcorp.com) handles MB (Aaron Mitchell).

2. **Manager**: 
   - All sales representatives (Jane Peacock, Steve Johnson, and Margaret Park) report to Nancy Edwards (nancy@chinookcorp.com), who is the single point of escalation for all the mentioned states and customers.
Gemini Response : The escalation matrix is organized as follows:

*   **Sales Representatives:**
    *   Jane Peacock (QC, BC, ON, NT)
    *   Steve Johnson (AB, NS)
    *   Margaret Park (MB)
*   **Manager:**
    *   Nancy Edwards (Manages all sales representatives listed)


**More Data**  
Data fetched for customers invoice amount and date for each Sales Rep  
Further this is captured in JSON

In [12]:
def group_json_by_n_columns(data: List[Dict[str, Any]], n: int) -> Dict:
    
    if not data:
        return {}

    # If n is 1, there is no need for re-aligning
    if n < 1:
        raise ValueError("n must be at least 1")

    # Check if those many columns are there
    column_keys = list(data[0].keys())
    if n > len(column_keys):
        raise ValueError(f"n = {n} exceeds number of columns = {len(column_keys)}")

    group_keys = column_keys[:n]

    def nest(rows: List[Dict[str, Any]], keys: List[str]) -> Dict:
        if not keys:
            return rows

        result = defaultdict(list)
        current_key = keys[0]

        for row in rows:
            group_value = row[current_key]
            remaining_row = {k: v for k, v in row.items() if k != current_key}
            result[group_value].append(remaining_row)

        # Recurse for each group
        return {
            k: nest(v, keys[1:]) for k, v in result.items()
        }

    return nest(data, group_keys)

In [13]:
# For Each Service Rep from Canada, their customer details, total sales per customer and the last purchase date 
Country = 'Canada'
result = conn_1.execute (text(f"""
                                SELECT
                                    E.FirstName || ' ' || E.LastName AS SupportRep,
                                    E.Email AS SupportEmail,
                                    E.Country AS Rep_Country,
                                    C.FirstName || ' ' || C.LastName AS Customer,
                                    C.Email AS CustomerEmail,
                                    SUM(I.Total) AS Total_Invoice,
                                    MAX(I.InvoiceDate) AS Last_Invoice
                                FROM
                                    Customer AS C
                                JOIN
                                    Invoice AS I ON C.CustomerId = I.CustomerId
                                JOIN
                                    Employee AS E ON C.SupportRepId = E.EmployeeId
                                WHERE
                                    E.Country = 'Canada'
                                GROUP BY    
                                    C.CustomerId
                                ORDER BY
                                    SupportRep, Customer;
                              """))

# Query output into JSON
rows = [row._asdict () for row in result]

# Re-align
result = group_json_by_n_columns(rows, 1)

# Convert to JSON
json_output = json.dumps(result,  indent=2)
print (json_output)

with open ('Output.json', 'w') as f:
    f.write (json_output)

{
  "Jane Peacock": [
    {
      "SupportEmail": "jane@chinookcorp.com",
      "Rep_Country": "Canada",
      "Customer": "Edward Francis",
      "CustomerEmail": "edfrancis@yachoo.ca",
      "Total_Invoice": 37.62,
      "Last_Invoice": "2011-01-02 00:00:00"
    },
    {
      "SupportEmail": "jane@chinookcorp.com",
      "Rep_Country": "Canada",
      "Customer": "Ellie Sullivan",
      "CustomerEmail": "ellie.sullivan@shaw.ca",
      "Total_Invoice": 37.62,
      "Last_Invoice": "2011-09-04 00:00:00"
    },
    {
      "SupportEmail": "jane@chinookcorp.com",
      "Rep_Country": "Canada",
      "Customer": "Emma Jones",
      "CustomerEmail": "emma_jones@hotmail.com",
      "Total_Invoice": 37.62,
      "Last_Invoice": "2011-06-11 00:00:00"
    },
    {
      "SupportEmail": "jane@chinookcorp.com",
      "Rep_Country": "Canada",
      "Customer": "Frank Ralston",
      "CustomerEmail": "fralston@gmail.com",
      "Total_Invoice": 43.62,
      "Last_Invoice": "2011-08-20 00:00:00"
 

In [14]:
Instr = "Using the context given, provide response to the user question or statement.\
        Context is in JSON format.\
        Provide Precise Answer.\
        Respond in bullet point"

# Question = f"When was the last purchase made by Jane's customer?"
# Question = f"Puja realised the invoice shall be 20.0. How much is the dispute amount? whom to contact?"
Question = f"Which service rep has cumulative customer invoice the highest?"

messages=[
    {
        "role": "system",
        "content": Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ json_output + "Query : \n" + Question
    }
]
completion = client_gr.chat.completions.create(
    messages=messages,
    model=model_gr,
    # temperature=0.0
)

print (completion.choices[0].message.content)

* To determine which service rep has the cumulative customer invoice the highest, we need to calculate the total invoice for each rep.
* Here are the total invoices for each rep:
  * Jane Peacock: 37.62 + 37.62 + 37.62 + 43.62 + 39.62 + 43.62 + 45.62 + 40.62 + 38.62 + 45.62 + 39.62 + 38.62 + 37.62 + 37.62 + 36.64 + 37.62 + 37.62 + 39.62 + 41.62 + 38.62 + 37.62 = **783.30**
  * Margaret Park: 37.62 + 39.62 + 38.62 + 37.62 + 39.62 + 37.62 + 37.62 + 37.62 + 37.62 + 40.62 + 39.62 + 37.62 + 37.62 + 37.62 + 47.62 + 37.62 = **604.86**
  * Steve Johnson: 37.62 + 42.62 + 37.62 + 37.62 + 49.62 + 39.62 + 38.62 + 40.62 + 43.62 + 37.62 + 37.62 + 37.62 + 46.62 + 37.62 + 37.62 + 37.62 + 37.62 + 42.62 = **673.86**
* Based on the calculations, **Jane Peacock** has the cumulative customer invoice the highest, with a total invoice of **783.30**.


>Numerical calculations across various parts of data, not to rely on LLM  
>Instead forumate the SQL to do the math part and use LLM to comprehend and respond

In [10]:
# For Service Rep in Canada, find the total invoice cumulative invoice for all their customers
result = conn_1.execute (text(f"""
                                SELECT
                                    E.FirstName || ' ' || E.LastName AS SupportRep,
                                    E.Email AS SupportEmail,
                                    E.Country AS Rep_Country,
                                    SUM(I.Total) AS Total_Invoice,
                                    MAX(I.InvoiceDate) AS Last_Invoice
                                FROM
                                    Customer AS C
                                JOIN
                                    Invoice AS I ON C.CustomerId = I.CustomerId
                                JOIN
                                    Employee AS E ON C.SupportRepId = E.EmployeeId
                                WHERE
                                    E.Country = 'Canada'
                                GROUP BY
                                    E.EmployeeId
                              """))

# Query output into JSON
rows = [row._asdict () for row in result]

# Re-align
result = group_json_by_n_columns(rows, 1)

# Convert to JSON
json_output = json.dumps(result,  indent=2)
print (json_output)

with open ('Output_summary.json', 'w') as f:
    f.write (json_output)

{
  "Jane Peacock": [
    {
      "SupportEmail": "jane@chinookcorp.com",
      "Rep_Country": "Canada",
      "Total_Invoice": 833.04,
      "Last_Invoice": "2011-12-22 00:00:00"
    }
  ],
  "Margaret Park": [
    {
      "SupportEmail": "margaret@chinookcorp.com",
      "Rep_Country": "Canada",
      "Total_Invoice": 775.4,
      "Last_Invoice": "2011-12-09 00:00:00"
    }
  ],
  "Steve Johnson": [
    {
      "SupportEmail": "steve@chinookcorp.com",
      "Rep_Country": "Canada",
      "Total_Invoice": 720.16,
      "Last_Invoice": "2011-12-05 00:00:00"
    }
  ]
}


In [11]:
Instr = "Using the context given, provide response to the user question or statement.\
        Context is in JSON format.\
        Respond in bullet point"

Question = f"Which service rep has cumulative customer invoice the highest?"

messages=[
    {
        "role": "system",
        "content": Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ json_output + "Query : \n" + Question
    }
]
completion = client_gr.chat.completions.create(
    messages=messages,
    model=model_gr,    
)

print (completion.choices[0].message.content)

* The service rep with the cumulative customer invoice the highest is Jane Peacock, with a total invoice of $833.04.
